In [71]:
import pandas as pd

train = pd.read_csv("../datasets/train.csv")
test  = pd.read_csv("../datasets/test.csv")
TARGET = '임신 성공 여부'
print(train.shape, test.shape)

(256351, 69) (90067, 68)


## 전처리 #1. 불필요 컬럼 제거
**방법:** `ID`, `난자 채취 경과일`, `불임 원인 - 여성 요인`, `정자 기증자 나이`, `난자 해동 경과일`, `난자 혼합 경과일` 제거

**이유:**
- `ID` → 식별자, 모델 입력 불필요
- `난자 채취 경과일`, `불임 원인 - 여성 요인` → 전체가 0.0인 상수 컬럼 (정보 없음)
- `정자 기증자 나이` → 기증 케이스 내 나이별 성공률 최대 4.4%p 차이 (만20세이하 20.9% ~ 알수없음 25.3%). 알수없음 제외 시 실질 차이 3.9%p 이하로 유의미한 정보 없음
- `난자 해동 경과일` → 99.4% 결측, 비결측 1,436건 중 99.9%가 0.0 (정보 없음)
- `난자 혼합 경과일` → 비결측 202,616건 중 99.7%가 0.0 (정보 없음)

> ⚠️ `착상 전 유전 검사 사용 여부`, `PGD 시술 여부`, `PGS 시술 여부`, `배아 해동 경과일`은 전처리 #2에서 이진 플래그 생성 후 원본 제거

**확인:** 상수 컬럼 min=max=std=0 확인 / 정자 기증자 나이 성공률 차이 확인 / 경과일 컬럼 값 분포 확인

In [72]:
# 상수 컬럼 확인 (min=max=std=0)
print("=== 상수 컬럼 확인 ===")
display(train[['난자 채취 경과일', '불임 원인 - 여성 요인']].describe().loc[['min', 'max', 'std']])

# 경과일 컬럼 값 분포 (비결측 중 0이 대부분인지 확인)
print("\n=== 경과일 컬럼 비결측 값 분포 ===")
for c in ['난자 해동 경과일', '난자 혼합 경과일']:
    non_null = train[c].dropna()
    zero_pct = (non_null == 0).mean() * 100
    print(f"[{c}] 비결측 {len(non_null):,}건, 0.0 비율: {zero_pct:.1f}%")

=== 상수 컬럼 확인 ===


,난자 채취 경과일,불임 원인 - 여성 요인
min,0.0,0.0
max,0.0,0.0
std,0.0,0.0



=== 경과일 컬럼 비결측 값 분포 ===
[난자 해동 경과일] 비결측 1,436건, 0.0 비율: 99.9%
[난자 혼합 경과일] 비결측 202,616건, 0.0 비율: 99.7%


In [73]:
# 정자 기증자 나이별 성공률 + 케이스 수 (차이 4%p 이하, 케이스 적음 → 제거 근거)
donor = train[train['정자 출처'] == '기증 제공'].groupby('정자 기증자 나이')[TARGET]
pd.DataFrame({
    '케이스 수': donor.count(),
    '성공률(%)': donor.mean().mul(100).round(2)
}).sort_values('성공률(%)', ascending=False)

,케이스 수,성공률(%)
정자 기증자 나이,,
알 수 없음,1534,25.29
만31-35세,4842,24.80
만26-30세,4986,24.73
만36-40세,5208,24.50
만41-45세,3798,24.14
만21-25세,5586,24.13
만20세 이하,1062,20.90


## 전처리 #2. 이진 플래그 생성 (원본 제거 포함)
**방법:** 결측이면 0, 값이 있으면 1인 이진 플래그 컬럼 신규 생성. 원본은 플래그 생성 후 제거

| 새 플래그 | 원본 컬럼 | 원본 처리 | 이유 |
|----------|----------|---------|------|
| `has_genetic_screening` | `착상 전 유전 검사 사용 여부` | 제거 | 값 있으면 항상 1.0 → 결측 여부 자체가 정보 |
| `has_pgd` | `PGD 시술 여부` | 제거 | 동일 |
| `has_pgs` | `PGS 시술 여부` | 제거 | 동일 |
| `has_embryo_thaw` | `배아 해동 경과일` | 제거 | 비결측 중 98.6%가 0.0 → 해동 실시 여부만 의미있음, 성공률 차이 확인 |
| `has_pregnancy_history` | `임신 시도 또는 마지막 임신 경과 연수` | **유지** | 플래그는 결측 여부, 원본 수치도 LightGBM이 NaN으로 처리 가능 → 정보 손실 없이 둘 다 활용 |

**확인:** PGD/PGS 항상 1.0 확인 / has_embryo_thaw 성공률 차이 확인 / has_pregnancy_history 성공률 차이 확인

In [74]:
# PGD/PGS/유전검사 — 값 있으면 항상 1 확인
cols = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']
pd.concat([
    train[cols].isnull().mean().mul(100).round(2).to_frame('결측률(%)').T,
    train[cols].describe().loc[['mean', 'min', 'max']]
])


,착상 전 유전 검사 사용 여부,PGD 시술 여부,PGS 시술 여부
결측률(%),98.94,99.15,99.25
mean,1.00,1.00,1.00
min,1.00,1.00,1.00
max,1.00,1.00,1.00


In [75]:
# 1. PGD/PGS/유전검사 — 값 있으면 항상 1.0 확인
cols = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']
display(pd.concat([
    train[cols].isnull().mean().mul(100).round(2).to_frame('결측률(%)').T,
    train[cols].describe().loc[['mean', 'min', 'max']]
]))

# 2. has_embryo_thaw — 해동 여부별 성공률 차이 확인 (플래그 근거)
train['has_embryo_thaw'] = train['배아 해동 경과일'].notna().astype(int)
print("\n=== has_embryo_thaw 성공률 차이 ===")
display(train.groupby('has_embryo_thaw')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2)))
train.drop(columns='has_embryo_thaw', inplace=True)

# 3. has_pregnancy_history — 결측 여부별 성공률 차이 확인 (플래그 근거)
train['has_pregnancy_history'] = train['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
print("\n=== has_pregnancy_history 성공률 차이 ===")
display(train.groupby('has_pregnancy_history')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2)))
train.drop(columns='has_pregnancy_history', inplace=True)

,착상 전 유전 검사 사용 여부,PGD 시술 여부,PGS 시술 여부
결측률(%),98.94,99.15,99.25
mean,1.00,1.00,1.00
min,1.00,1.00,1.00
max,1.00,1.00,1.00



=== has_embryo_thaw 성공률 차이 ===


,성공률,샘플 수
has_embryo_thaw,,
0,26.39,215982
1,22.86,40369



=== has_pregnancy_history 성공률 차이 ===


,성공률,샘플 수
has_pregnancy_history,,
0,25.99,246981
1,21.68,9370


## 전처리 #3. 문자열 → 정수 변환 (횟수 컬럼 10개)
**방법:** `str.extract(r'(\d+)').astype(int)` — 앞의 숫자만 추출 (`"6회 이상"` → `6`)

**이유:** 횟수는 실제로 순서가 있는 값 (0 < 1 < 2 < ...). 트리 모델은 분기점 기준으로 처리하므로 사이값 문제 없음. 범주형 One-hot보다 컬럼 수 절약 및 순서 정보 보존

**확인:** 10개 컬럼 unique 값 목록으로 패턴 및 최댓값 확인

In [76]:
hoecols = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수'
]
{c: sorted(train[c].unique().tolist()) for c in hoecols}

{'총 시술 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 '클리닉 내 총 시술 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 'IVF 시술 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 'DI 시술 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 '총 임신 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 'IVF 임신 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 'DI 임신 횟수': ['0회', '1회', '2회', '3회', '4회', '5회'],
 '총 출산 횟수': ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상'],
 'IVF 출산 횟수': ['0회', '1회', '2회', '3회', '4회', '5회'],
 'DI 출산 횟수': ['0회', '1회', '2회', '3회', '5회']}

## 전처리 #4. 순서형 인코딩 (시술 당시 나이, 난자 기증자 나이)
**방법:**
- `시술 당시 나이`: 만18-34세→0, 만35-37세→1, 만38-39세→2, 만40-42세→3, 만43-44세→4, 만45-50세→5, 알 수 없음→NaN
- `난자 기증자 나이`: 만20세 이하→1, 만21-25세→2, 만26-30세→3, 만31-35세→4, 알 수 없음→NaN

**이유:**
- 나이는 순서가 있는 범주형. 트리 모델은 숫자의 크기로 분기하므로 순서 정보를 활용 가능
- `알 수 없음`은 NaN으로 처리 → LightGBM이 자체적으로 최적 방향으로 배치
- 난자 기증자 나이는 기증 케이스(~6%)에서만 유의미하며 나이별 성공률 차이 약 10%p 확인됨 (만26-30세 34.8% ~ 만20세이하 26.2%)

**주의:** 난자 기증자 나이의 성공률이 단조감소하지 않음 (만26-30세 최고, 만20세 이하 낮음). 트리 모델은 임의 분기점 사용으로 비단조 관계도 처리 가능하므로 순서형 인코딩 유효

**확인:** unique 값 / 나이별 성공률 / `알 수 없음` 329건 성공률 0% 원인 조사

In [77]:
# 1. unique 값 + 나이별 성공률 확인
for c in ['시술 당시 나이', '난자 기증자 나이']:
    print(f"{c}: {sorted(train[c].unique().tolist())}")

print("\n=== 시술 당시 나이별 성공률 ===")
display(train.groupby('시술 당시 나이')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2))
        .sort_values('성공률', ascending=False))

print("\n=== 난자 기증자 나이별 성공률 (기증 케이스만) ===")
display(train[train['난자 출처'] == '기증 제공'].groupby('난자 기증자 나이')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2))
        .sort_values('성공률', ascending=False))

# 2. '알 수 없음' 329건 성공률 0% 원인 조사
print("\n=== 시술 당시 나이 알 수 없음 — 시술 유형 분포 ===")
unknown_age = train[train['시술 당시 나이'] == '알 수 없음']
print(f"총 {len(unknown_age)}건")
display(unknown_age[['시술 유형', '특정 시술 유형', TARGET]].value_counts().head(10).to_frame('건수'))

시술 당시 나이: ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세', '알 수 없음']
난자 기증자 나이: ['만20세 이하', '만21-25세', '만26-30세', '만31-35세', '알 수 없음']

=== 시술 당시 나이별 성공률 ===


,성공률,샘플 수
시술 당시 나이,,
만18-34세,32.26,102476
만35-37세,27.84,57780
만38-39세,21.71,39247
만45-50세,16.77,6918
만40-42세,15.94,37348
만43-44세,11.80,12253
알 수 없음,0.00,329



=== 난자 기증자 나이별 성공률 (기증 케이스만) ===


,성공률,샘플 수
난자 기증자 나이,,
만26-30세,34.83,4976
만21-25세,32.99,2334
만31-35세,30.52,6366
만20세 이하,26.19,294
알 수 없음,25.07,1799



=== 시술 당시 나이 알 수 없음 — 시술 유형 분포 ===
총 329건


건수
시술 유형 특정 시술 유형  임신 성공 여부     
IVF   IVF       0         310
      ICSI      0          11
      ICSI:ICSI 0           6
      Unknown   0           1
      IVF:IVF   0           1

## 전처리 #5. 이진 인코딩 (시술 유형: IVF→1, DI→0)
**방법:** `map({'IVF': 1, 'DI': 0})`

**이유:** unique 값이 2개뿐이므로 One-hot 불필요. IVF(체외수정)와 DI(인공수정)는 시술 방식이 근본적으로 다르며 성공률 차이도 확인 필요

**확인:** value_counts로 unique 값 2개 확인 + 시술 유형별 성공률 차이 확인

In [ ]:
# 시술 유형 분포 + 성공률 차이 확인 (이진 인코딩 근거)
print("=== 시술 유형 분포 ===")
display(train['시술 유형'].value_counts().to_frame('건수'))

print("\n=== 시술 유형별 성공률 ===")
display(train.groupby('시술 유형')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2)))

## 전처리 #6. One-hot 인코딩 (배란 유도 유형, 난자 출처, 정자 출처)
**방법:** 소수 카테고리 병합 후 `pd.get_dummies(drop_first=True)`

**소수 카테고리 병합 방침:**
- `배란 유도 유형`: `세트로타이드 (억제제)` 1건, `생식선 자극 호르몬` 1건 → `기록되지 않은 시행`으로 병합 (임상적으로 기록 누락과 동일하게 처리, One-hot 시 분리해봐야 학습 불가 수준)
- `정자 출처`: `미할당` 122건, `배우자 및 기증 제공` 14건 → `기증 제공`으로 병합 (비배우자 정자 사용 케이스로 분류)
- `난자 출처`: 소수 카테고리 없음 (`알 수 없음` 6,291건은 DI 케이스로 유지)

**이유:** unique 수가 3~4개로 낮아 One-hot 시 추가 피처 수 2~3개로 부담 없음. 단, 1~122건 수준의 극소수 카테고리는 학습 샘플 부족으로 별도 컬럼화 시 과적합 위험

> LightGBM은 `category` dtype으로 범주형을 직접 처리할 수도 있으나, One-hot이 명시적이고 다른 모델(XGBoost 등)과 호환성이 높아 선택

**확인:** 카테고리별 케이스 수 확인 (병합 대상 소수 카테고리 식별)

In [79]:
# 카테고리별 케이스 수 확인 (소수 카테고리 존재 여부)
for c in ['배란 유도 유형', '난자 출처', '정자 출처']:
    print(f"\n=== {c} ===")
    display(train[c].value_counts().to_frame('건수').assign(비율=lambda d: (d['건수']/len(train)*100).round(2)))


=== 배란 유도 유형 ===


,건수,비율
배란 유도 유형,,
기록되지 않은 시행,194432,75.85
알 수 없음,61917,24.15
세트로타이드 (억제제),1,0.00
생식선 자극 호르몬,1,0.00



=== 난자 출처 ===


,건수,비율
난자 출처,,
본인 제공,234291,91.39
기증 제공,15769,6.15
알 수 없음,6291,2.45



=== 정자 출처 ===


,건수,비율
정자 출처,,
배우자 제공,229199,89.41
기증 제공,27016,10.54
미할당,122,0.05
배우자 및 기증 제공,14,0.01


## 전처리 #7. Target Encoding (특정 시술 유형, 시술 시기 코드) + 멀티 레이블 플래그 (배아 생성 주요 이유)

**방법:**
- `특정 시술 유형`, `시술 시기 코드` → `category_encoders.TargetEncoder(smoothing=10)`
  - 학습 데이터로만 fit, 테스트에 transform 적용 (데이터 누수 방지)
  - smoothing=10 으로 소수 케이스 범주를 전체 평균 쪽으로 보정
- `배아 생성 주요 이유` → 쉼표 구분 멀티 레이블을 이진 플래그 5개로 분리
  - `이유_현재시술용`, `이유_기증용`, `이유_난자저장용`, `이유_배아저장용`, `이유_연구용`

**이유:**
- `특정 시술 유형`: 24개 카테고리 그대로 유지 (추가 처치 정보 유의미 — BLASTOCYST 여부로 성공률 8~10%p 차이)
  소수 케이스(1~6건) 범주 존재 → 스무딩으로 과적합 방지
- `시술 시기 코드`: 의미 불명 코드값 → 분리 불가, 타겟 성공률로 대체
- `배아 생성 주요 이유`: 쉼표로 묶인 복합값 → Target Encoding 시 조합별로 다른 값으로 취급되어 정보 손실, 이진 플래그가 적합

**확인:** 카테고리별 케이스 수 + 성공률, 멀티 레이블 기본 범주 확인

In [80]:
# 1. 특정 시술 유형 — 케이스 수 + 성공률 (소수 케이스 스무딩 근거)
print("=== 특정 시술 유형 ===")
display(train.groupby('특정 시술 유형')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2))
        .sort_values('성공률', ascending=False))

# 2. 시술 시기 코드 — 코드별 성공률 차이 확인 (Target Encoding 적용 근거)
print("\n=== 시술 시기 코드별 성공률 ===")
display(train.groupby('시술 시기 코드')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': '성공률', 'count': '샘플 수'})
        .assign(성공률=lambda d: d['성공률'].mul(100).round(2))
        .sort_values('성공률', ascending=False))

# 3. 배아 생성 주요 이유 — 멀티 레이블 기본 범주 확인
print("\n=== 배아 생성 주요 이유 unique 값 ===")
print(sorted(train['배아 생성 주요 이유'].dropna().unique().tolist()))

=== 특정 시술 유형 ===


,성공률,샘플 수
특정 시술 유형,,
ICSI / BLASTOCYST :ICSI,100.00,1
ICSI / AH:Unknown,50.00,2
IVF / BLASTOCYST,36.62,1248
ICSI / BLASTOCYST,35.67,1609
FER,33.33,3
ICSI / BLASTOCYST:IVF / BLASTOCYST,33.33,6
IVF:ICSI,28.32,392
ICSI,27.28,122368
IVF,26.15,91755



=== 시술 시기 코드별 성공률 ===


,성공률,샘플 수
시술 시기 코드,,
TRYBLT,26.91,36713
TRJXFG,26.63,36031
TRVNRY,25.98,36173
TRCMWS,25.74,38090
TRXQMD,25.61,34831
TRZKPL,25.51,35544
TRDQAZ,24.54,38969



=== 배아 생성 주요 이유 unique 값 ===
['기증용', '기증용, 난자 저장용', '기증용, 배아 저장용', '기증용, 배아 저장용, 현재 시술용', '기증용, 현재 시술용', '난자 저장용', '난자 저장용, 배아 저장용', '난자 저장용, 배아 저장용, 연구용', '난자 저장용, 현재 시술용', '배아 저장용', '배아 저장용, 현재 시술용', '연구용, 현재 시술용', '현재 시술용']


## 전처리 #8. float → int 변환 (단일 배아 이식 여부, 착상 전 유전 진단 사용 여부)
**방법:** 결측치를 0으로 채운 후 `astype(int)` (또는 결측 유지 시 `pd.Int8Dtype()` 사용)

**이유:** 실제로 0.0과 1.0만 존재하는 이진값이지만 결측치로 인해 float64로 저장됨. 의미상 이진 플래그이므로 정수형이 적합

**결측 처리 근거 (0으로 채우는 이유):**
- 두 컬럼 모두 NaN 6,291건 = 전처리 #5의 DI 케이스 수(6,291건)와 일치
- DI(인공수정)는 배아 이식 절차 자체가 없으므로 `단일 배아 이식 여부`와 `착상 전 유전 진단 사용 여부`가 해당 없음(0)인 것이 임상적으로 타당
- 따라서 NaN → 0 처리는 도메인 지식에 근거한 채우기이며 정보 손실 없음

**주의:**
- NaN이 있는 상태에서 일반 `astype(int)` 사용 시 에러 발생 → 반드시 결측 처리 후 변환
- LightGBM은 float NaN도 처리 가능하므로 굳이 변환하지 않아도 무방. 가독성/명시성 목적의 변환

**확인:** dtype=float64 확인 / unique 값 0.0·1.0만 존재 확인 / 결측률 확인 / NaN 행이 DI 케이스임을 검증

In [ ]:
cols = ['단일 배아 이식 여부', '착상 전 유전 진단 사용 여부']
for c in cols:
    vc = train[c].value_counts(dropna=False)
    total = len(train)
    print(f"\n[{c}] dtype={train[c].dtype}")
    print(vc.rename({0.0: '0 (아니오)', 1.0: '1 (예)', float('nan'): 'NaN'})
          .to_frame('건수')
          .assign(비율=lambda d: (d['건수']/total*100).round(2)))

# NaN 행이 DI 케이스인지 검증 (0 채우기 근거)
print("\n=== NaN 행의 시술 유형 분포 ===")
nan_mask = train['단일 배아 이식 여부'].isna()
print(f"NaN 건수: {nan_mask.sum():,}건 / DI 케이스 수: {(train['시술 유형']=='DI').sum():,}건")
print(train.loc[nan_mask, '시술 유형'].value_counts().to_string())
print("→ NaN = DI 케이스와 정확히 일치 → 0으로 채우기 타당")

## 전처리 #9. 결측치 처리 — NaN 유지 (LightGBM 자체 처리)
**방법:** 전처리 #1·#2에서 처리되지 않은 나머지 결측치는 NaN 그대로 유지

**이유:** LightGBM/XGBoost는 NaN을 분기 시 최적 방향으로 자동 배치. 무리하게 채우면 오히려 잘못된 정보를 주입할 수 있음

**잔여 결측 컬럼 처리 방침:**
- `배아 이식 경과일` (17%) → NaN 유지. 2~5일 다양한 분포로 유의미한 수치
- `배아 관련 수치 14개` (~2.45%) → NaN 유지. LightGBM 처리
- `임신 시도 또는 마지막 임신 경과 연수` (96%) → NaN 유지 + `has_pregnancy_history` 플래그 (#2에서 처리)

**확인:** 전처리 #1·#2 처리 후 남은 결측 컬럼 목록 및 결측률

In [82]:
# 전처리 #1·#2에서 처리된 컬럼 제외 후 남은 결측 컬럼 확인
removed_cols = [
    'ID', '난자 채취 경과일', '불임 원인 - 여성 요인', '정자 기증자 나이',
    '난자 해동 경과일', '난자 혼합 경과일',
    '착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부', '배아 해동 경과일'
]
remaining = train.drop(columns=removed_cols, errors='ignore')
missing = remaining.isnull().mean().mul(100).round(2)
display(missing[missing > 0].sort_values(ascending=False).to_frame('결측률(%)'))
print(f"\n결측 있는 컬럼 수: {(missing > 0).sum()}개")

,결측률(%)
임신 시도 또는 마지막 임신 경과 연수,96.34
배아 이식 경과일,16.99
단일 배아 이식 여부,2.45
배아 생성 주요 이유,2.45
총 생성 배아 수,2.45
미세주입된 난자 수,2.45
착상 전 유전 진단 사용 여부,2.45
미세주입에서 생성된 배아 수,2.45
이식된 배아 수,2.45
저장된 배아 수,2.45



결측 있는 컬럼 수: 23개


## 전처리 #10. 클래스 불균형 처리 — class_weight='balanced'
**방법:** LightGBM `class_weight='balanced'` 옵션 사용 (또는 `is_unbalance=True`)

**이유:** 타겟이 74:26으로 불균형. 그대로 학습하면 모델이 다수 클래스(실패)만 예측하는 방향으로 편향됨

**확인:** 타겟 샘플 수 및 비율

In [83]:
train[TARGET].value_counts().rename({0:'실패', 1:'성공'}).to_frame('샘플 수').assign(비율=lambda d: (d['샘플 수'] / d['샘플 수'].sum() * 100).round(2))

,샘플 수,비율
임신 성공 여부,,
실패,190123,74.17
성공,66228,25.83
